In [11]:
%cd /content/RiskAware-Complaints-Engine
!git pull origin master
!ls scripts

/content/RiskAware-Complaints-Engine
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.33 KiB | 795.00 KiB/s, done.
From https://github.com/Nusultan11/RiskAware-Complaints-Engine
 * branch            master     -> FETCH_HEAD
   590fafc..30799e4  master     -> origin/master
Updating 590fafc..30799e4
Fast-forward
 pyproject.toml                  |   1 +
 scripts/tune_category_optuna.py | 137 ++++++++++++++++++++++++++++++++++++++++
 2 files changed, 138 insertions(+)
 create mode 100644 scripts/tune_category_optuna.py
error_analysis_category.py  prepare_data.py    tune_category_optuna.py
evaluate_category.py	    train_category.py  tune_category.py


In [20]:
import os, json, zipfile,shutil, glob
from pathlib import Path
import yaml
import pandas as pd
from google.colab import files as gfiles
from google.colab import files
from pathlib import Path

In [4]:
os.makedirs("/content/RiskAware-Complaints-Engine/data/raw", exist_ok=True)
uploaded = files.upload()

Saving cfpb_complaints.csv to cfpb_complaints.csv


In [8]:
print("cwd:", os.getcwd())
print("matches:", glob.glob("/content/**/cfpb_complaints.csv", recursive=True))

cwd: /content/RiskAware-Complaints-Engine
matches: ['/content/RiskAware-Complaints-Engine/cfpb_complaints.csv']


In [9]:
src_candidates = glob.glob("/content/**/cfpb_complaints.csv", recursive=True)
src = [p for p in src_candidates if "/data/raw/" not in p][0]

dst = "/content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv"
os.makedirs("/content/RiskAware-Complaints-Engine/data/raw", exist_ok=True)
shutil.copy(src, dst)

print("copied:", src, "->", dst)
print("exists:", os.path.exists(dst))

copied: /content/RiskAware-Complaints-Engine/cfpb_complaints.csv -> /content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv
exists: True


In [13]:
os.environ["PYTHONPATH"] = "src"

In [15]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.6 MB/s eta 0:00:00


In [18]:
!python src/risk_aware/data/prepare.py
!python scripts/tune_category_optuna.py --n-trials 40 --timeout-sec 7200
!cat reports/metrics/category_optuna_best.json

[I 2026-03-19 09:24:36,837] A new study created in memory with name: no-name-1c76484d-d0e9-4157-a16d-3e2609f780dd
Starting Optuna tuning: n_trials=40, timeout_sec=7200
[I 2026-03-19 09:46:34,551] Trial 0 finished with value: 0.25244731254765435 and parameters: {'max_features': 70000, 'min_df': 2, 'max_df': 0.85, 'c': 8.605201659336437, 'class_weight': None}. Best is trial 0 with value: 0.25244731254765435.
[I 2026-03-19 09:54:38,483] Trial 1 finished with value: 0.2839074789994489 and parameters: {'max_features': 70000, 'min_df': 1, 'max_df': 0.95, 'c': 3.6838954793622753, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.2839074789994489.
[I 2026-03-19 10:01:51,479] Trial 2 finished with value: 0.28323117556803024 and parameters: {'max_features': 50000, 'min_df': 2, 'max_df': 0.9, 'c': 3.58882401195222, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.2839074789994489.
[I 2026-03-19 10:07:47,123] Trial 3 finished with value: 0.2739037468357054 and parameters: {'max_

In [21]:
best_path = Path("reports/metrics/category_optuna_best.json")
with best_path.open("r", encoding="utf-8") as f:
    best = json.load(f)
best_params = best["params"]
print("BEST:", best)

BEST: {'trial_number': 4, 'macro_f1': 0.2869974276468556, 'accuracy': 0.5218981810791586, 'n_features': 70000, 'fit_time_sec': 532.783, 'params': {'max_features': 70000, 'min_df': 3, 'max_df': 0.9, 'c': 3.641226167642757, 'class_weight': 'balanced'}}


In [22]:
cfg_path = Path("configs/category.yaml")
with cfg_path.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["stack"] = "tfidf_lr"
cfg["stacks"]["tfidf_lr"]["max_features"] = int(best_params["max_features"])
cfg["stacks"]["tfidf_lr"]["ngram_range"] = [1, 2]
cfg["stacks"]["tfidf_lr"]["min_df"] = int(best_params["min_df"])
cfg["stacks"]["tfidf_lr"]["max_df"] = float(best_params["max_df"])
cfg["stacks"]["tfidf_lr"]["c"] = float(best_params["c"])
cfg["stacks"]["tfidf_lr"]["class_weight"] = best_params["class_weight"]

with cfg_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Updated configs/category.yaml with best params")

Updated configs/category.yaml with best params


In [23]:
!PYTHONPATH=src python scripts/train_category.py
!PYTHONPATH=src python scripts/evaluate_category.py

Category training completed.
Macro-F1: 0.2870
Accuracy: 0.5219
Artifacts saved to: artifacts/category
Validation metrics saved to: reports/metrics/category_metrics_val.json
category_macro_f1: 0.297802
Saved metrics to: reports/metrics/category_metrics_test.json


In [24]:
bundle_name = "riskaware_tuned_bundle.zip"
paths_to_pack = [
    "artifacts/category",
    "reports/metrics/category_metrics_val.json",
    "reports/metrics/category_metrics_test.json",
    "reports/metrics/category_optuna_best.json",
    "reports/metrics/category_optuna_trials.csv",
    "data/processed",
    "configs/category.yaml",
]

with zipfile.ZipFile(bundle_name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths_to_pack:
        pth = Path(p)
        if pth.is_file():
            z.write(pth, arcname=str(pth))
        elif pth.is_dir():
            for file in pth.rglob("*"):
                if file.is_file():
                    z.write(file, arcname=str(file))

print("Created:", bundle_name)

Created: riskaware_tuned_bundle.zip


In [25]:
gfiles.download(bundle_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>